# NS-3 Assignment 7 Analysis

Run the simulations first, then execute this notebook from the assignment folder.

```bash
./ns3 run 'tutdgr --config=scratch/topology.json --outputDir=output/data/dgr'
./ns3 run 'tutrip --config=scratch/topology.json --outputDir=output/data/rip'
```

The code below also accepts a copied `data/dgr` and `data/rip` directory, matching the submission layout requested in the assignment.

In [ ]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 4.8)
plt.rcParams['axes.grid'] = True

base = Path('data') if Path('data').exists() else Path('output/data')
dgr_dir = base / 'dgr'
rip_dir = base / 'rip'
print('Using data directory:', base.resolve())

In [ ]:
def parse_ns3_time(value):
    if value is None:
        return 0.0
    match = re.match(r'([+-]?[0-9.]+)([a-z]+)', value)
    if not match:
        return float(value)
    number = float(match.group(1))
    unit = match.group(2)
    scale = {'s': 1.0, 'ms': 1e-3, 'us': 1e-6, 'ns': 1e-9, 'ps': 1e-12}
    return number * scale[unit]

def parse_flowmon(path):
    root = ET.parse(path).getroot()
    rows = []
    for flow in root.findall('./FlowStats/Flow'):
        attrs = flow.attrib
        rx = int(attrs.get('rxPackets', 0))
        rows.append({
            'flowId': int(attrs['flowId']),
            'txPackets': int(attrs.get('txPackets', 0)),
            'rxPackets': rx,
            'lostPackets': int(attrs.get('lostPackets', 0)),
            'delaySum': parse_ns3_time(attrs.get('delaySum', '0s')),
            'jitterSum': parse_ns3_time(attrs.get('jitterSum', '0s')),
            'timeFirstTx': parse_ns3_time(attrs.get('timeFirstTxPacket', '0s')),
            'timeLastRx': parse_ns3_time(attrs.get('timeLastRxPacket', '0s')),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df['avgDelay'] = df.apply(lambda r: r.delaySum / r.rxPackets if r.rxPackets else 0.0, axis=1)
        df['avgJitter'] = df.apply(lambda r: r.jitterSum / max(r.rxPackets - 1, 1) if r.rxPackets else 0.0, axis=1)
    return df

def load_timeseries(folder):
    path = folder / 'flow-stats-timeseries.csv'
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def load_rtt(folder):
    path = folder / 'ping-rtt.csv'
    return pd.read_csv(path) if path.exists() else pd.DataFrame(columns=['time', 'rtt', 'context'])

In [ ]:
dgr_flow = parse_flowmon(dgr_dir / 'flowmon.xml')
rip_flow = parse_flowmon(rip_dir / 'flowmon.xml')
dgr_ts = load_timeseries(dgr_dir)
rip_ts = load_timeseries(rip_dir)
dgr_rtt = load_rtt(dgr_dir)
rip_rtt = load_rtt(rip_dir)

summary = pd.DataFrame({
    'scheme': ['DGR', 'RIP'],
    'flows': [len(dgr_flow), len(rip_flow)],
    'rxPackets': [dgr_flow.rxPackets.sum(), rip_flow.rxPackets.sum()],
    'lostPackets': [dgr_flow.lostPackets.sum(), rip_flow.lostPackets.sum()],
    'avgPacketDelay_s': [dgr_flow.delaySum.sum() / dgr_flow.rxPackets.sum(), rip_flow.delaySum.sum() / rip_flow.rxPackets.sum()],
    'avgPacketJitter_s': [dgr_flow.jitterSum.sum() / max(dgr_flow.rxPackets.sum() - len(dgr_flow), 1), rip_flow.jitterSum.sum() / max(rip_flow.rxPackets.sum() - len(rip_flow), 1)],
})
summary

## Flow interpretation

The flow monitor should normally report three flows: the UDP OnOff traffic from `src` to `dst`, the ICMP echo requests from `src` to `dst`, and the ICMP echo replies from `dst` to `src`. The exact packet counts differ between DGR and RIP because the link failure at 50 s and recovery at 300 s trigger different route repair behavior.

In [ ]:
def plot_rx_loss(ts, title):
    if ts.empty:
        print('No time series data for', title)
        return
    total = ts.groupby('time')[['rxPackets', 'lostPackets', 'droppedPackets']].sum().reset_index()
    ax = total.plot(x='time', y=['rxPackets', 'lostPackets', 'droppedPackets'])
    ax.axvline(50, color='tab:red', linestyle='--', label='a-d down')
    ax.axvline(300, color='tab:green', linestyle='--', label='a-d up')
    ax.set_title(title)
    ax.set_xlabel('Simulation time [s]')
    ax.set_ylabel('Packets')
    ax.legend()
    plt.show()

plot_rx_loss(dgr_ts, 'DGR received/lost/dropped packets over time')
plot_rx_loss(rip_ts, 'RIP received/lost/dropped packets over time')

Packet loss is expected around the failure and, depending on the protocol, during convergence afterwards. DGR recomputes routes using global topology knowledge and should therefore converge faster. RIP is distributed; routing updates need time to propagate, and transient stale routes can keep packets on an invalid path for longer.

In [ ]:
def plot_rtt(df, title):
    if df.empty:
        print('No RTT data for', title)
        return
    ax = df.plot(x='time', y='rtt', legend=False)
    ax.axvline(50, color='tab:red', linestyle='--')
    ax.axvline(300, color='tab:green', linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Simulation time [s]')
    ax.set_ylabel('RTT [s]')
    plt.show()

plot_rtt(dgr_rtt, 'DGR ping RTT')
plot_rtt(rip_rtt, 'RIP ping RTT')

The RTT is lower before the `a-d` failure because the direct path is used. During the failure, traffic must use the longer `a-b-c-d` path once routing has converged. After 300 s, the direct path becomes available again and the RTT should fall back toward the original value. RIP may show a wider transition region than DGR because its view of the topology changes through routing updates rather than an immediate global recomputation.

In [ ]:
comparison = summary.set_index('scheme')[['lostPackets', 'avgPacketJitter_s']]
comparison.plot(kind='bar', subplots=True, layout=(1, 2), legend=False, figsize=(10, 4))
plt.tight_layout()
plt.show()
comparison

DGR is expected to lose fewer packets and show lower jitter than RIP. The main reason is convergence: DGR reacts to interface events by recomputing global routes, while RIP relies on distributed periodic/triggered updates. Jitter comes from route changes, queuing while traffic shifts to a longer path, and transient losses/recoveries around the failed link.

In [ ]:
def changed_route_tables(route_file):
    text = Path(route_file).read_text(errors='ignore')
    blocks = []
    current = []
    for line in text.splitlines():
        if line.startswith('Node:') and current:
            blocks.append('\n'.join(current))
            current = [line]
        else:
            current.append(line)
    if current:
        blocks.append('\n'.join(current))

    changed = []
    previous_body = None
    for block in blocks:
        body = '\n'.join(line for line in block.splitlines() if not line.startswith('Node:'))
        if body != previous_body:
            changed.append(block)
            previous_body = body
    return changed

for path in sorted(rip_dir.glob('routes-*.txt')):
    changes = changed_route_tables(path)
    print('\n====', path.name, 'changed', len(changes), 'times ====')
    for block in changes[:6]:
        print(block[:1200])
        print('---')

The RIP routing tables should change after the failed `a-d` interface is taken down and again after it is restored. The important entries are the routes to the host networks `10.1.0.0/24` and `10.1.1.0/24`, plus the backbone alternatives through `b` and `c`. The tables change gradually because routers learn updated distances from neighbors instead of sharing one instantaneous global topology view.